# Zajęcia 2 — powtarzalne środowisko i szkielet projektu

Eksperyment z zajęć 1 zamieniamy w projekt: strukturę katalogów, konfigurację poza kodem, moduły do zaimportowania i jedną komendę uruchamiającą całość. Do tego dokładamy trzy rzeczy, bez których projekt nie jest powtarzalny: **blokadę zależności**, **kontrolę stylu** i **poprawny `.gitignore`**.

Część dotycząca Gita i GitHuba wykonywana jest **poza notebookiem** — notebook przygotowuje pliki i sprawdza, czy niczego nie brakuje, ale nie tworzy repozytorium za ciebie.

Komórki opisane jako **zadanie** wypełniasz samodzielnie; komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`. Zajrzyj do niego dopiero po własnej próbie.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian. Sprawdza też, które narzędzia są dostępne; ich brak nie przerywa pracy, tylko pomija odpowiednie kroki.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PROJECT_DIR = REPO_ROOT / "artifacts" / "zajecia_02" / "projekt"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


def write_file(relative_path: str, content: str) -> Path:
    path = PROJECT_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print(f"Zapisano: {relative_path}")
    return path


def tool(name: str):
    """Szuka narzedzia w PATH, a potem obok interpretera tego srodowiska."""
    found = shutil.which(name)
    if found:
        return found
    candidate = Path(sys.executable).parent / (name + (".exe" if os.name == "nt" else ""))
    return str(candidate) if candidate.exists() else None


TOOLS = {name: tool(name) for name in ("uv", "ruff", "git")}
for name, path in TOOLS.items():
    print(f"{name:5s} {'dostępne' if path else 'BRAK — kroki zostaną pominięte'}")

%load_ext skip_kernel_extension

## 2. Struktura katalogów

Struktura ma rozdzielać cztery rzeczy o różnym cyklu życia: kod, konfigurację, dane i artefakty.

**Zadanie:** utwórz katalogi projektu. Zastanów się, dlaczego `data/` i `artifacts/` są w tym zestawie, skoro i tak nie trafią do repozytorium.

In [ ]:
# Zadanie: utwórz katalogi projektu.
SUBDIRS = ["config", "src", "data", "artifacts/models", "artifacts/reports", "notebooks"]


In [ ]:
%%skip True
for subdir in SUBDIRS:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "src" / "__init__.py").touch()

for subdir in SUBDIRS:
    print(f"  {subdir}/")
print("\nKatalogi data/ i artifacts/ istnieją w projekcie, ale nie w repozytorium —")
print("kod musi umieć je odtworzyć, a nie zakładać, że ktoś je przyniósł.")

## 3. Konfiguracja w pliku YAML

Parametr zapisany w kodzie jest niewidoczny i trudny do zmiany. Wyprowadzamy do pliku wszystko, co wpływa na wynik eksperymentu.

**Zadanie:** uzupełnij `config/config.yaml` o sekcje `data`, `split`, `model` i `paths`. Kryterium: po zmianie samego tego pliku ma dać się uruchomić inny eksperyment, bez dotykania kodu.

In [ ]:
# Zadanie: uzupełnij plik konfiguracyjny.
write_file(
    "config/config.yaml",
    """
    seed: 42
    """,
)

In [ ]:
%%skip True
write_file(
    "config/config.yaml",
    """
    # Parametry eksperymentu. Wszystko, co wplywa na wynik, jest tutaj.
    seed: 42

    data:
      raw_path: data/iris.csv
      target_column: target

    split:
      test_size: 0.3
      stratify: true

    model:
      name: logistic_regression
      max_iter: 500
      C: 1.0

    paths:
      model: artifacts/models/model.joblib
      metrics: artifacts/reports/metrics.json
    """,
)

## 4. Moduł wczytujący konfigurację

Obowiązuje hierarchia źródeł, od najsłabszego do najsilniejszego:

1. wartość domyślna w kodzie,
2. plik konfiguracyjny,
3. zmienna środowiskowa,
4. argument wiersza poleceń.

**Zadanie:** napisz `src/config.py` z funkcją `load_config(path)`, która wczytuje plik YAML i pozwala nadpisać `seed` zmienną środowiskową `PROJEKT_SEED`. Zwróć obiekt, do którego łatwo się odwołać.

In [ ]:
# Zadanie: napisz moduł konfiguracji.


In [ ]:
%%skip True
write_file(
    "src/config.py",
    """
    # Wczytanie konfiguracji z hierarchia zrodel:
    # domyslne -> plik -> zmienne srodowiskowe.
    from __future__ import annotations

    import os
    from dataclasses import dataclass, field
    from pathlib import Path

    import yaml

    ENV_PREFIX = "PROJEKT_"


    @dataclass
    class Config:
        seed: int = 42
        data: dict = field(default_factory=dict)
        split: dict = field(default_factory=dict)
        model: dict = field(default_factory=dict)
        paths: dict = field(default_factory=dict)


    def load_config(path: str | Path = "config/config.yaml") -> Config:
        raw = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}

        # Zmienna srodowiskowa jest silniejsza niz plik - to pozwala
        # uruchomic ten sam kod inaczej, bez modyfikacji repozytorium.
        env_seed = os.environ.get(ENV_PREFIX + "SEED")
        if env_seed is not None:
            raw["seed"] = int(env_seed)

        known = {"seed", "data", "split", "model", "paths"}
        unknown = sorted(set(raw) - known)
        if unknown:
            raise ValueError(f"Nieznane klucze konfiguracji: {unknown}")

        return Config(**raw)
    """,
)

## 5. Moduł potoku

Trzy funkcje o jawnym wejściu i wyjściu: `prepare`, `train`, `evaluate`. Każda przyjmuje konfigurację i zapisuje wynik na dysk — bez stanu ukrytego w pamięci notebooka.

**Zadanie:** napisz `src/pipeline.py`. `prepare` ma zapisać dane wejściowe do pliku, `train` — model, `evaluate` — metryki w formacie JSON.

In [ ]:
# Zadanie: napisz potok: prepare, train, evaluate.


In [ ]:
%%skip True
write_file(
    "src/pipeline.py",
    """
    # Trzy kroki potoku. Kazdy ma jawne wejscie, jawne wyjscie i zaden stan ukryty.
    from __future__ import annotations

    import json
    from pathlib import Path

    import joblib
    import pandas as pd
    from sklearn.datasets import load_iris
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    from src.config import Config


    def prepare(config: Config) -> Path:
        destination = Path(config.data["raw_path"])
        destination.parent.mkdir(parents=True, exist_ok=True)
        load_iris(as_frame=True).frame.to_csv(destination, index=False)
        print(f"prepare: dane zapisane w {destination}")
        return destination


    def _split(config: Config):
        frame = pd.read_csv(config.data["raw_path"])
        target = config.data["target_column"]
        features = [column for column in frame.columns if column != target]
        return train_test_split(
            frame[features],
            frame[target],
            test_size=config.split["test_size"],
            random_state=config.seed,
            stratify=frame[target] if config.split["stratify"] else None,
        )


    def train(config: Config) -> Path:
        x_train, _, y_train, _ = _split(config)
        model = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(
                        max_iter=config.model["max_iter"],
                        C=config.model["C"],
                        random_state=config.seed,
                    ),
                ),
            ]
        ).fit(x_train, y_train)

        destination = Path(config.paths["model"])
        destination.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, destination)
        print(f"train: model zapisany w {destination}")
        return destination


    def evaluate(config: Config) -> Path:
        _, x_test, _, y_test = _split(config)
        model = joblib.load(config.paths["model"])
        predictions = model.predict(x_test)

        metrics = {
            "seed": config.seed,
            "accuracy": round(float(accuracy_score(y_test, predictions)), 6),
            "f1_macro": round(float(f1_score(y_test, predictions, average="macro")), 6),
            "n_test": int(len(y_test)),
        }

        destination = Path(config.paths["metrics"])
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
        print(f"evaluate: {metrics}")
        return destination
    """,
)

## 6. Interfejs wiersza poleceń

Potok CI ani harmonogram nie umieją „uruchomić wszystkich komórek". Potrzebują komendy.

**Zadanie:** napisz `src/cli.py` obsługujący cele `prepare`, `train`, `evaluate` i `all`, z opcją `--config`.

In [ ]:
# Zadanie: napisz interfejs wiersza poleceń.


In [ ]:
%%skip True
write_file(
    "src/cli.py",
    """
    # Jedyny sposob uruchomienia projektu. To samo wywolanie stosuje czlowiek i CI.
    from __future__ import annotations

    import argparse

    from src.config import load_config
    from src.pipeline import evaluate, prepare, train

    STEPS = {"prepare": prepare, "train": train, "evaluate": evaluate}


    def main() -> None:
        parser = argparse.ArgumentParser(prog="iris-projekt")
        parser.add_argument("target", choices=[*STEPS, "all"])
        parser.add_argument("--config", default="config/config.yaml")
        arguments = parser.parse_args()

        config = load_config(arguments.config)
        targets = list(STEPS) if arguments.target == "all" else [arguments.target]
        for name in targets:
            STEPS[name](config)


    if __name__ == "__main__":
        main()
    """,
)

## 7. `Makefile`, `README.md` i `.gitignore`

Trzy pliki, które decydują o tym, czy ktoś inny w ogóle uruchomi projekt. Ta komórka jest gotowa — uruchom ją bez zmian.

Zwróć uwagę na `.gitignore`: nie jest to formalność, tylko lista czterech kategorii, które nie mają prawa trafić do repozytorium.

In [ ]:
write_file(
    "Makefile",
    """
    # Zadania projektowe. Ten sam interfejs dla czlowieka i dla CI.
    PYTHON ?= python

    prepare:
\t$(PYTHON) -m src.cli prepare

    train:
\t$(PYTHON) -m src.cli train

    evaluate:
\t$(PYTHON) -m src.cli evaluate

    all:
\t$(PYTHON) -m src.cli all

    .PHONY: prepare train evaluate all
    """,
)

write_file(
    "README.md",
    """
    # Iris — projekt demonstracyjny

    Klasyfikacja gatunku irysa na podstawie czterech pomiarów.

    ## Uruchomienie

    ```bash
    uv sync
    uv run python -m src.cli all
    ```

    Bez `uv`:

    ```bash
    pip install -r requirements.txt
    python -m src.cli all
    ```

    ## Wynik

    - `artifacts/models/model.joblib` — wytrenowany model
    - `artifacts/reports/metrics.json` — metryki na zbiorze testowym

    ## Konfiguracja

    Parametry eksperymentu: `config/config.yaml`.
    Ziarno losowości można nadpisać zmienną `PROJEKT_SEED`.
    """,
)

write_file(
    ".gitignore",
    """
    # dane - rozmiar, licencje, dane osobowe
    data/

    # artefakty - odtwarzalne z kodu i danych
    artifacts/
    *.joblib

    # srodowisko - zalezy od maszyny
    .venv/
    __pycache__/
    *.py[cod]
    .ipynb_checkpoints/

    # sekrety
    .env
    """,
)

## 8. Uruchomienie jedną komendą

Uruchamiamy projekt **osobnym procesem**, a nie importem w notebooku. To jedyny sposób, żeby sprawdzić, że działa naprawdę, a nie dzięki zmiennym, które akurat są w pamięci jądra.

**Zadanie:** napisz `run_project(target, extra_env=None)` uruchamiającą `python -m src.cli <target>` w katalogu projektu i zwracającą wynik procesu.

In [ ]:
# Zadanie: uruchom potok w osobnym procesie.


In [ ]:
%%skip True
def run_project(target: str, extra_env: dict = None) -> subprocess.CompletedProcess:
    environment = {**os.environ, **(extra_env or {})}
    result = subprocess.run(
        [sys.executable, "-m", "src.cli", target],
        cwd=PROJECT_DIR,
        env=environment,
        capture_output=True,
        text=True,
    )
    print(result.stdout.strip() or result.stderr.strip()[-1200:])
    if result.returncode != 0:
        raise RuntimeError(f"Cel '{target}' zakończył się kodem {result.returncode}")
    return result


run_project("all")

## 9. Sprawdzenie powtarzalności

Dwa uruchomienia tego samego kodu z tą samą konfiguracją muszą dać identyczne metryki. Jeżeli nie dają, gdzieś została nieustalona losowość.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
METRICS_PATH = PROJECT_DIR / "artifacts" / "reports" / "metrics.json"

run_project("all")
first = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
run_project("all")
second = json.loads(METRICS_PATH.read_text(encoding="utf-8"))

print(f"\nIdentyczne metryki: {first == second}")
display(pd.DataFrame([first, second], index=["przebieg 1", "przebieg 2"]))

## 10. Punkt kontrolny 1 — inne ziarno losowości

Uruchamiamy ten sam kod z inną wartością `PROJEKT_SEED`. Nic w repozytorium się nie zmienia.

**Do opisania:** które metryki się zmieniły i dlaczego? Czy to znaczy, że model jest niestabilny?

In [ ]:
results = {"seed=42": first}
for seed in ("7", "2024"):
    run_project("all", extra_env={"PROJEKT_SEED": seed})
    results[f"seed={seed}"] = json.loads(METRICS_PATH.read_text(encoding="utf-8"))

display(pd.DataFrame(results).T)

run_project("all")
print("\nPrzywrócono ziarno z pliku konfiguracyjnego.")

## 11. Deklaracja i blokada zależności

Dwa poziomy opisu zależności:

| Poziom | Co zawiera | Plik |
|---|---|---|
| **deklaracja** | biblioteki, których projekt faktycznie używa | `pyproject.toml` |
| **blokada** | dokładne wersje **wszystkich** zależności, także pośrednich | `uv.lock` |

Deklaracja opisuje intencję, blokada gwarantuje powtarzalność.

**Zadanie:** zapisz `pyproject.toml` z zależnościami projektu, wygeneruj blokadę poleceniem `uv lock` i przypnij wersję interpretera przez `uv python pin`.

In [ ]:
# Zadanie: zapisz deklarację zależności i wygeneruj blokadę.


In [ ]:
%%skip True
write_file(
    "pyproject.toml",
    """
    [project]
    name = "iris-projekt"
    version = "0.1.0"
    description = "Projekt demonstracyjny do zajec z technik MLOps"
    requires-python = ">=3.12,<3.13"
    dependencies = [
        "joblib>=1.4",
        "pandas>=2.2",
        "PyYAML>=6.0",
        "scikit-learn>=1.5",
    ]

    [dependency-groups]
    dev = ["pytest>=8.0", "ruff>=0.6"]

    # Projekt jest aplikacja, nie biblioteka - nie budujemy z niego pakietu.
    [tool.uv]
    package = false
    """,
)

if TOOLS["uv"] is None:
    print("\nBrak narzędzia uv — pomijam generowanie blokady.")
    print("Wykonaj samodzielnie w katalogu projektu:  uv lock  oraz  uv python pin 3.12")
else:
    for arguments in (["lock"], ["python", "pin", "3.12"]):
        result = subprocess.run(
            [TOOLS["uv"], *arguments], cwd=PROJECT_DIR, capture_output=True, text=True
        )
        print(f"uv {' '.join(arguments)}: {(result.stdout + result.stderr).strip().splitlines()[-1]}")

    lock_path = PROJECT_DIR / "uv.lock"
    if lock_path.exists():
        locked = lock_path.read_text(encoding="utf-8").count("[[package]]")
        print(f"\nZadeklarowano bibliotek: 4")
        print(f"Zablokowano pakietów:    {locked}")
        print(f"Wersja interpretera:     {(PROJECT_DIR / '.python-version').read_text().strip()}")

## 12. Punkt kontrolny 2 — co naprawdę instalujesz

Różnica między liczbą zadeklarowanych bibliotek a liczbą pakietów w blokadzie to **zależności pośrednie** — kod, którego nikt świadomie nie wybierał, a który mimo to znajdzie się w twoim środowisku i w twoim obrazie.

**Do opisania:** ile pakietów pojawiło się dodatkowo i co z tego wynika dla powtarzalności oraz dla bezpieczeństwa? Do tej listy wrócimy na zajęciach 9, gdy będziemy ją skanować.

In [ ]:
import re

lock_path = PROJECT_DIR / "uv.lock"
if not lock_path.exists():
    print("Brak pliku uv.lock — pomijam porównanie.")
else:
    declared = {
        re.split(r"[<>=!~ ]", line.strip().strip('",'))[0].lower()
        for line in (PROJECT_DIR / "pyproject.toml").read_text(encoding="utf-8").splitlines()
        if line.strip().startswith('"') and ">=" in line
    }
    locked = set(re.findall(r'^name = "(.+)"', lock_path.read_text(encoding="utf-8"), re.MULTILINE))
    locked.discard("iris-projekt")
    transitive = sorted(locked - declared)

    print(f"Zadeklarowane ({len(declared)}): {', '.join(sorted(declared))}")
    print(f"\nDociągnięte pośrednio ({len(transitive)}):")
    print("  " + ", ".join(transitive))
    print(
        f"\nNa jedną świadomie wybraną bibliotekę przypada "
        f"{len(transitive) / max(len(declared), 1):.1f} pakietu, o którym nikt nie decydował."
    )

## 13. Kontrola stylu

Spór o formatowanie jest najtańszym sposobem na zmarnowanie czasu zespołu. Rozstrzyga się go raz — konfiguracją w repozytorium.

`ruff` pełni dwie role: wykrywa błędy (nieużywane importy, przesłonięte nazwy, martwy kod) i formatuje kod.

**Zadanie:** zapisz `ruff.toml`, dopisz do projektu plik z typowymi usterkami, uruchom kontrolę, a następnie napraw automatycznie to, co da się naprawić.

In [ ]:
# Zadanie: skonfiguruj ruff i sprawdź projekt.


In [ ]:
%%skip True
write_file(
    "ruff.toml",
    """
    # Konfiguracja nalezy do repozytorium, nie do maszyny.
    # Bez niej wynik zalezy od wersji narzedzia u danej osoby.
    line-length = 100

    [lint]
    select = ["E4", "E7", "E9", "F", "I"]
    """,
)

# Plik z typowymi usterkami: nieuzywane importy, zly porzadek, martwa zmienna.
write_file(
    "src/scratch.py",
    """
    import sys
    import json
    from pathlib import Path

    import pandas as pd


    def summarize(values):
        unused = 123
        return {"n": len(values), "max": max(values)}
    """,
)


def ruff(*arguments) -> subprocess.CompletedProcess:
    return subprocess.run(
        [TOOLS["ruff"], *arguments], cwd=PROJECT_DIR, capture_output=True, text=True
    )


if TOOLS["ruff"] is None:
    print("Brak narzędzia ruff — pomijam kontrolę stylu.")
else:
    before = ruff("check", "src")
    print("Przed naprawą:")
    print(before.stdout.strip()[-1200:] or before.stderr.strip()[-600:])

    ruff("check", "--fix", "src")
    ruff("format", "src")

    after = ruff("check", "src")
    print("\nPo naprawie:")
    print(after.stdout.strip() or "(brak zgłoszeń)")
    print(f"\nKod wyjścia: {after.returncode}")
    print(
        "\nCzego ruff NIE naprawił samodzielnie: nieużywanej zmiennej lokalnej.\n"
        "Narzędzie nie zgaduje intencji — usunięcie kodu zostawia człowiekowi."
    )

## 14. Hak przed commitem i kontrola `.gitignore`

Hak `pre-commit` uruchamia te same narzędzia przed każdym commitem. Kosztuje sekundy, a oszczędza przebieg CI i cudzy czas.

Druga rzecz do sprawdzenia przed pierwszym commitem to `.gitignore` — bo pliku raz wysłanego do repozytorium nie da się z niego usunąć zwykłym `git rm`.

**Zadanie:** zapisz `.pre-commit-config.yaml` oraz napisz `check_gitignore()` sprawdzającą, czy plik obejmuje cztery kategorie: dane, artefakty, środowisko i sekrety.

In [ ]:
# Zadanie: skonfiguruj hak przed commitem i sprawdź .gitignore.


In [ ]:
%%skip True
write_file(
    ".pre-commit-config.yaml",
    """
    # Instalacja w repozytorium:  pre-commit install
    # Pierwsze uruchomienie pobiera narzedzia, wiec wymaga sieci.
    # Wartosc `rev` warto co jakis czas odswiezyc: pre-commit autoupdate
    repos:
      - repo: https://github.com/astral-sh/ruff-pre-commit
        rev: v0.16.6
        hooks:
          - id: ruff
            args: [--fix]
          - id: ruff-format

      - repo: https://github.com/pre-commit/pre-commit-hooks
        rev: v5.0.0
        hooks:
          - id: end-of-file-fixer
          - id: trailing-whitespace
          - id: check-yaml
          - id: check-added-large-files
            args: [--maxkb=500]
    """,
)

REQUIRED_PATTERNS = {
    "dane": ("data/",),
    "artefakty": ("artifacts/", "*.joblib"),
    "środowisko": (".venv/", "__pycache__/"),
    "sekrety": (".env",),
}


def check_gitignore(path: Path) -> list:
    if not path.exists():
        return ["brak pliku .gitignore"]
    entries = {
        line.strip()
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.strip().startswith("#")
    }
    return [
        f"{category}: brak wpisu {' albo '.join(patterns)}"
        for category, patterns in REQUIRED_PATTERNS.items()
        if not any(pattern in entries for pattern in patterns)
    ]


problems = check_gitignore(PROJECT_DIR / ".gitignore")
if problems:
    for problem in problems:
        print(f"  BRAK: {problem}")
else:
    print(".gitignore obejmuje wszystkie cztery kategorie.")

print("\nSprawdzenie na pliku niekompletnym:")
incomplete = PROJECT_DIR / ".gitignore.probka"
incomplete.write_text("__pycache__/\n", encoding="utf-8")
for problem in check_gitignore(incomplete):
    print(f"  BRAK: {problem}")
incomplete.unlink()

## 15. Punkt kontrolny 3 — świeży klon

Usuwamy wszystko, czego nie ma w repozytorium: dane i artefakty. Zostaje dokładnie to, co dostałby ktoś po `git clone`.

**Do opisania:** czy projekt odbudował się jedną komendą? Jeżeli nie — czego zabrakło?

In [ ]:
shutil.rmtree(PROJECT_DIR / "data", ignore_errors=True)
shutil.rmtree(PROJECT_DIR / "artifacts", ignore_errors=True)
print(f"Po usunięciu — data/ istnieje: {(PROJECT_DIR / 'data').exists()}")
print(f"Po usunięciu — artifacts/ istnieje: {(PROJECT_DIR / 'artifacts').exists()}\n")

run_project("all")

restored = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
print(f"\nOdbudowano: {METRICS_PATH.exists()}")
print(f"Metryki identyczne jak przed usunięciem: {restored == first}")

## 16. Poza notebookiem — repozytorium zdalne

Tę część wykonaj w terminalu, w katalogu **własnego projektu zaliczeniowego**, nie w katalogu demonstracyjnym.

### Krok 1 — repozytorium lokalne

```bash
git init
git add .
git status          # sprawdź, czy nie ma tu data/, artifacts/ ani .env
git commit -m "Szkielet projektu"
git branch -M main
```

Krok `git status` przed pierwszym commitem jest istotny. Sekret usunięty kolejnym commitem **nadal jest w historii** — usunięcie go wymaga przepisania historii i unieważnienia poświadczenia.

### Krok 2 — repozytorium zdalne

Utwórz puste repozytorium na GitHubie (bez `README`, bez `.gitignore` — masz już własne), a potem:

```bash
git remote add origin https://github.com/<uzytkownik>/<projekt>.git
git push -u origin main
```

### Krok 3 — hak przed commitem

```bash
pre-commit install
pre-commit run --all-files
```

Pierwsze uruchomienie pobiera narzędzia i trwa dłużej. Kolejne są niemal natychmiastowe.

### Krok 4 — sprawdzenie z drugiej strony

Sklonuj własne repozytorium do innego katalogu i wykonaj instrukcję z `README.md`. Jeżeli musisz cokolwiek dopowiedzieć — `README.md` jest niekompletny.

```bash
git clone https://github.com/<uzytkownik>/<projekt>.git sprawdzenie
cd sprawdzenie
uv sync
uv run python -m src.cli all
```

## 17. Podsumowanie

Projekt ma strukturę rozdzielającą kod, konfigurację, dane i artefakty; parametry poza kodem; ustalone ziarno; blokadę zależności z przypiętą wersją interpretera; kontrolę stylu; hak przed commitem i `.gitignore`, który faktycznie sprawdziliśmy. Uruchamia się jedną komendą i odbudowuje po usunięciu wszystkiego, czego nie ma w repozytorium.

Czego nadal brakuje:

- dane pobierane są za każdym razem od nowa i nikt nie wie, czy to wciąż te same dane (zajęcia 3);
- nie ma kontraktu opisującego, jak dane mają wyglądać (zajęcia 3);
- kroki potoku wykonują się zawsze, nawet gdy nic się nie zmieniło (zajęcia 4).

### Zadanie do własnego projektu

1. `README.md` z opisem problemu, użytkownika systemu i metryki sukcesu.
2. Struktura katalogów rozdzielająca kod, konfigurację, dane i artefakty.
3. `pyproject.toml` + `uv.lock` + `.python-version`.
4. Parametry eksperymentu w pliku konfiguracyjnym, nie w kodzie.
5. Ustalone ziarna losowości i wykazana powtarzalność dwóch przebiegów.
6. Jedna udokumentowana komenda uruchamiająca całość.
7. `.gitignore` obejmujący dane, artefakty, środowisko i sekrety.
8. `ruff.toml` w repozytorium i zainstalowany hak `pre-commit`.
9. Projekt wysłany na GitHub i sprawdzony przez sklonowanie do innego katalogu.

**Kryterium ukończenia:** inna osoba klonuje repozytorium, wykonuje instrukcję z `README.md` i otrzymuje plik z metrykami bez zadawania pytań.